# Lesson 4 — Full project: Linear Regression vs Neural Network 🧪

You will build **two complete machine-learning models** using the same Advertising Sales data, and compare them fairly.

**Prerequisites:** Lessons 1–3. Every step explains why it exists.

**What you'll do:** load data → explore → split → scale → train linear regression → train neural network → choose neural network using validation data → compare on untouched test data → visualize → reflect.

> The dataset is small (200 rows) and observational. A more complex model is **not automatically better**, and relationships are not proof that advertising caused sales.

## Step 1 — Import tools and load the dataset

`X` means **features** (inputs). `y` means **target** (what we want to predict).

We have three features: TV, Radio, and Newspaper advertising budgets. We want to predict Sales.

In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

csv_path = Path('data/Advertising.csv')
if not csv_path.exists():
    csv_path = Path('../data/Advertising.csv')
df = pd.read_csv(csv_path).drop(columns=['Unnamed: 0'], errors='ignore')
assert list(df.columns) == ['TV', 'Radio', 'Newspaper', 'Sales']
print('Rows:',len(df))
df.head()

## Step 2 — Look at the data before modeling

Always inspect data first. `describe` gives count, average, spread and minimum/maximum. The charts let us see whether each budget is associated with Sales.

In [ ]:
display(df.describe().round(2))
print('Missing values in each column:\n', df.isna().sum())
fig, axes = plt.subplots(1,3,figsize=(12,3.4))
for ax, feature in zip(axes,['TV','Radio','Newspaper']):
    ax.scatter(df[feature],df['Sales'],alpha=0.65)
    ax.set(xlabel=feature,ylabel='Sales',title=f'{feature} vs Sales')
plt.tight_layout()
plt.show()

## Step 3 — Keep the test set untouched

We need three groups:

- **Training (60%)**: where both models learn.
- **Validation (20%)**: where we decide when to stop neural-network training.
- **Test (20%)**: our final fair exam, used only at the very end.

We use the same split for **both** models to make comparison meaningful.

In [ ]:
X = df[['TV','Radio','Newspaper']]
y = df['Sales']
X_build, X_test, y_build, y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_build,y_build,test_size=0.25,random_state=SEED)
print('Training:',len(X_train),'Validation:',len(X_val),'Test:',len(X_test))
assert len(X_train)==120 and len(X_val)==40 and len(X_test)==40

## Step 4 — Scale using the training set only

**Scaling** makes values comparable: a high TV budget should not dominate just because its numbers are larger. We `fit` the scaler on training inputs and only `transform` validation and test inputs. This avoids **data leakage** (peeking at information from the exam).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print('One scaled training row:',X_train_scaled[0].round(2))

## Step 5 — Model A: Linear regression (simple baseline)

This learns one straight-line-style formula combining the three features:

**Sales ≈ weight1 × TV + weight2 × Radio + weight3 × Newspaper + bias**

`fit` learns the weights from training data. `predict` uses the learned formula on new inputs.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled,y_train)
print('Learned weights:',dict(zip(X.columns,linear_model.coef_.round(3))))
print('Learned bias:',round(linear_model.intercept_,3))
linear_val_pred = linear_model.predict(X_val_scaled)
print('Validation MAE:',round(mean_absolute_error(y_val,linear_val_pred),3))

## Step 6 — Model B: Neural network

A neural network can learn nonlinear patterns using extra layers and an **activation function**.

- `Linear(3, 16)`: three input features, 16 hidden values
- `ReLU()`: lets the model learn bends, not only straight lines
- `Linear(16, 1)`: one Sales prediction

`torch.float32` is the number format that PyTorch uses here.

In [ ]:
to_tensor = lambda values: torch.tensor(np.asarray(values),dtype=torch.float32)
X_train_t = to_tensor(X_train_scaled)
X_val_t = to_tensor(X_val_scaled)
y_train_t = to_tensor(y_train.to_numpy().reshape(-1,1))
y_val_t = to_tensor(y_val.to_numpy().reshape(-1,1))
X_test_t = to_tensor(X_test_scaled)

neural_model = nn.Sequential(
    nn.Linear(3,16),
    nn.ReLU(),
    nn.Linear(16,1)
)
loss_fn = nn.MSELoss()  # Mean squared error: lower is better
optimizer = torch.optim.Adam(neural_model.parameters(), lr=0.01)
print(neural_model)

## Step 7 — Train and use validation to stop

Each **epoch** is one pass over the training examples. A **forward pass** makes predictions. `loss.backward()` calculates gradients (how the weights should change). `optimizer.step()` adjusts the weights.

**Early stopping:** If validation loss does not improve for 100 epochs, we stop and restore the best weights. Importantly, we do **not** use the test set for this decision.

In [ ]:
train_losses, val_losses = [], []
best_val_loss = float('inf')
best_weights = None
best_epoch = 0
patience = 100
stale_epochs = 0

for epoch in range(1500):
    neural_model.train()
    train_prediction = neural_model(X_train_t)
    train_loss = loss_fn(train_prediction,y_train_t)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    neural_model.eval()
    with torch.no_grad():
        val_loss = loss_fn(neural_model(X_val_t),y_val_t).item()
    train_losses.append(train_loss.item())
    val_losses.append(val_loss)

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(neural_model.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1
    if stale_epochs >= patience:
        break

neural_model.load_state_dict(best_weights)
neural_model.eval()
print('Best epoch:',best_epoch,'| Best validation MSE:',round(best_val_loss,3))

## Step 8 — Draw the learning curve

If training loss goes down but validation loss goes up, that is a sign of **overfitting**: memorizing the training patterns instead of learning to generalize.

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(train_losses,label='Training MSE')
plt.plot(val_losses,label='Validation MSE')
plt.axvline(best_epoch-1,linestyle='--',color='gray',label='Best validation epoch')
plt.xlabel('Epoch')
plt.ylabel('Mean squared error')
plt.title('Neural network learning curve')
plt.legend()
plt.show()

## Step 9 — The final fair comparison (test set)

Only now do we evaluate on the **unseen test examples**.

**MAE:** average absolute error (smaller is better). **RMSE:** error metric that penalizes large mistakes (smaller is better). **R²:** how much variation the model explains compared with a mean-only baseline (higher is better, may be negative).

We also compare with a simple **mean baseline** that predicts the same training-average Sales for every row. This tells us whether the models do more than a naive guess.

In [ ]:
linear_test_pred = linear_model.predict(X_test_scaled)
with torch.no_grad():
    neural_test_pred = neural_model(X_test_t).numpy().ravel()
mean_test_pred = np.full(len(y_test),y_train.mean())

def score(name,pred):
    return {
        'Model':name,
        'MAE ↓':mean_absolute_error(y_test,pred),
        'RMSE ↓':np.sqrt(mean_squared_error(y_test,pred)),
        'R² ↑':r2_score(y_test,pred)
    }

comparison = pd.DataFrame([
    score('Predict training mean',mean_test_pred),
    score('Linear regression',linear_test_pred),
    score('Neural network',neural_test_pred)
]).set_index('Model')
display(comparison.round(3))
print('Best by test MAE:',comparison['MAE ↓'].idxmin())

## Step 10 — View predictions side by side

Perfect predictions lie on the dotted diagonal. The closer points are to this diagonal, the better.

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(11,4.2))
for ax,name,pred in zip(axes,['Linear regression','Neural network'],[linear_test_pred,neural_test_pred]):
    ax.scatter(y_test,pred,alpha=0.8)
    lo=min(y_test.min(),pred.min())
    hi=max(y_test.max(),pred.max())
    ax.plot([lo,hi],[lo,hi],'--',color='gray')
    ax.set(xlabel='Actual Sales',ylabel='Predicted Sales',title=name)
plt.tight_layout()
plt.show()

preview = pd.DataFrame({'Actual':y_test.to_numpy(),'Linear':linear_test_pred,'Neural network':neural_test_pred})
display(preview.head(10).round(2))

## Step 11 — Your own conclusion ✍️

Write your answers before moving on:

1. What are the **features**, and what is the **target**?
2. Why do we have separate training, validation and test sets?
3. Which model did best by **MAE**? Does it beat the training-mean baseline?
4. What did the neural-network **loss curve** do? Did validation loss rise?
5. Why might a neural network lose to linear regression on such a small dataset?

### Optional experiments
- Change hidden neurons from `16` to `8`, **then restart and run all cells** (to reset random seeds), and compare validation loss.
- Try learning rate `0.001` instead of `0.01`.
- For serious comparisons, make modeling decisions using **validation**, then evaluate your chosen model on test data only once. If you try lots of variants against the test set, it stops being a fair final exam.

**Congratulations!** You built two models, used a validation set, measured accuracy, and compared predictions end to end.